<a href="https://colab.research.google.com/github/jawariyakhawer/nlp/blob/main/nlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

Using device: cpu


In [ ]:
# Step 1: Imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import urllib.request
import tarfile
import os

# Step 2: IMDB dataset download karo (chota sample use karenge speed ke liye)
print("Downloading dataset...")
url = "https://raw.githubusercontent.com/Ankit152/IMDB-sentiment-analysis/master/IMDB-Dataset.csv"
urllib.request.urlretrieve(url, "imdb.csv")
print("Downloaded!")

# Step 3: Data load karo
df = pd.read_csv("imdb.csv")
print("Total reviews:", len(df))
print(df.head())

# Speed ke liye sirf 5000 reviews use karenge (CPU friendly)
df = df.sample(5000, random_state=42)

# Step 4: Labels ko convert karo (positive=1, negative=0)
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# Step 5: Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    df['review'], df['label'], test_size=0.2, random_state=42
)

# Step 6: Text ko numbers mein convert karo (TF-IDF)
print("Converting text to features...")
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Step 7: Model train karo (Logistic Regression - CPU pe bahut fast)
print("Training model...")
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

# Step 8: Test karo
predictions = model.predict(X_test_vec)
accuracy = accuracy_score(y_test, predictions)
print(f"\nTest Accuracy: {accuracy*100:.2f}%")
print("\n", classification_report(y_test, predictions, target_names=['negative', 'positive']))

# Step 9: Apna khud ka review test karo
def predict_sentiment(text):
    vec = vectorizer.transform([text])
    pred = model.predict(vec)[0]
    prob = model.predict_proba(vec)[0]
    sentiment = "Positive" if pred == 1 else "Negative"
    confidence = max(prob) * 100
    print(f"Review: {text}")
    print(f"Sentiment: {sentiment} (Confidence: {confidence:.2f}%)")

predict_sentiment("This movie was absolutely amazing, I loved every minute of it!")
predict_sentiment("Terrible movie, waste of time, worst acting ever.")

Downloaded!
Total reviews: 50000
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
Converting text to features...
Training model...

Test Accuracy: 84.60%

               precision    recall  f1-score   support

    negative       0.88      0.81      0.84       506
    positive       0.82      0.88      0.85       494

    accuracy                           0.85      1000
   macro avg       0.85      0.85      0.85      1000
weighted avg       0.85      0.85      0.85      1000

Review: This movie was absolutely amazing, I loved every minute of it!
Sentiment: Positive (Confidence: 89.34%)
Review: Terrible movie, waste of time, worst acting ever.
Sentiment: Negativ